In [ ]:
import numpy as np
from sklearn.datasets import make_moons
import seaborn as sns
from mlxtend.plotting import plot_decision_regions
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense, Dropout, Conv2D, Flatten, MaxPooling2D

from keras.optimizers import Adam

In [ ]:
train_data = tf.keras.utils.image_dataset_from_directory(
    directory="..\\..\\..\\cnn\\train",      # Path to the training data directory
    labels="inferred",                      # Infer labels from subdirectory names
    label_mode="int",                       # Labels are returned as integers
    batch_size=32,                          # Number of images per batch
    image_size=(256, 256),                  # Resize images to 256x256 pixels
)

In [ ]:
test_data = tf.keras.utils.image_dataset_from_directory(
    directory="..\\..\\..\\cnn\\test",
    labels="inferred",
    label_mode="int",
    batch_size=32,
    image_size=(256, 256),
)

In [ ]:

# normalize the data so that the pixel values are between 0 and 1
def normalize(image, label):
    """
    Normalizes image pixel values to the range [0, 1].

    Args:
        image: A tensor representing the image, with pixel values in the range [0, 255].
        label: The corresponding label for the image.

    Returns:
        A tuple of (normalized_image, label), where normalized_image has pixel values in the range [0, 1].
    """
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

In [ ]:
train_data = train_data.map(normalize)
test_data = test_data.map(normalize)

### Define CNN Model

In [ ]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001), input_shape=(256, 256, 3)),
    MaxPooling2D((2, 2), strides=2),
    Conv2D(64, (3, 3), activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    MaxPooling2D((2, 2), strides=2),
    Conv2D(128, (3, 3), activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    MaxPooling2D((2, 2), strides=2),
    Flatten(),
    Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    Dropout(0.3),
    Dense(64, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

In [ ]:
model.summary()

In [ ]:
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [ ]:
history = model.fit(train_data,
                    epochs=10,
                    validation_data=test_data,
                    verbose=1,
                    use_multiprocessing=True,
                    workers=6)

In [ ]:
plt.plot(history.history['accuracy'], color='red', label='train accuracy')
plt.plot(history.history['val_accuracy'], color='blue', label='test accuracy')
plt.title('Model Accuracy')
plt.legend()
plt.show()

In [ ]:
plt.plot(history.history['loss'], color='red', label='training loss')
plt.plot(history.history['val_loss'], color='blue', label='validation loss')
plt.title('Model Loss')
plt.legend()
plt.show()

In [ ]:
import cv2
test_img = cv2.imread('y1.jpg')
plt.imshow(test_img)

In [ ]:
test_img.shape

In [ ]:
test_img = cv2.resize(test_img, (256, 256))

In [ ]:
test_input = test_img.reshape((1, 256, 256, 3))

In [ ]:
model.predict(test_input)